In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from torch.optim import AdamW
from langdetect import detect
import re
import nltk
from tqdm import tqdm
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=False)

nltk.download("cmudict")

from nltk.corpus import cmudict
cmu_dict = cmudict.dict()

In [ ]:
############################################
# PARAMETERS
############################################

CSV_PATH = "/kaggle/input/datasets/abhaysharma01702/cleaned-spotify-dataset"
MODEL_SAVE_PATH = "./emotion_gpt2_lyrics"
EPOCHS = 3
BATCH_SIZE = 8
MAX_LEN = 128
LR = 5e-5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print(DEVICE)

In [ ]:
############################################
# LOAD DATASET
############################################

import pandas as pd

df = pd.read_csv(r"D:\Natural Language\NLP\Project\Datasets\spotify_cleaned_dataset.csv")

print("Original dataset size:", len(df))
print("\nOriginal distribution:")
print(df["emotion"].value_counts())


############################################
# BALANCED SAMPLING
############################################

# target max samples per emotion
TARGET_PER_CLASS = 40000   # change if you want

balanced_parts = []

for emotion, group in df.groupby("emotion"):

    if len(group) > TARGET_PER_CLASS:
        sampled = group.sample(TARGET_PER_CLASS, random_state=42)
    else:
        sampled = group   # keep all if smaller

    balanced_parts.append(sampled)

balanced_df = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)


############################################
# RESULTS
############################################

print("\nBalanced dataset size:", len(balanced_df))
print("\nBalanced distribution:")
print(balanced_df["emotion"].value_counts())

texts = balanced_df.apply(lambda x: f"<{x['emotion']}> {x['text']}", axis=1).tolist()

In [ ]:
############################################
# TOKENIZER
############################################

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 has no padding token → use EOS token
tokenizer.pad_token = tokenizer.eos_token

# add emotion tokens
tokenizer.add_special_tokens({
    "additional_special_tokens":[
        "<happy>",
        "<sad>",
        "<anger>",
        "<fear>",
        "<surprise>",
        "<neutral>"
    ]
})



In [ ]:
############################################
# DATASET (LAZY TOKENIZATION)
############################################

from torch.utils.data import Dataset, DataLoader

class LyricsDataset(Dataset):

    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": encoding["input_ids"].squeeze()
        }


############################################
# CREATE DATASET
############################################

dataset = LyricsDataset(texts, tokenizer, MAX_LEN)

############################################
# DATALOADER (MULTI-CORE)
############################################

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,      # parallel CPU workers
    pin_memory=True     # faster GPU transfer
)

In [ ]:
############################################
# LOAD MODEL
############################################

model = GPT2LMHeadModel.from_pretrained("gpt2")

model.resize_token_embeddings(len(tokenizer))

model.to(DEVICE)

optimizer = AdamW(model.parameters(),lr=LR)

In [ ]:
############################################
# TRAINING WITH CHECKPOINTS
############################################

import os

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model.train()

for epoch in range(EPOCHS):

    total_loss = 0

    for step, batch in enumerate(loader):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # OPTIONAL: save checkpoint every N steps
        if step % 5000 == 0 and step > 0:

            ckpt_path = f"{CHECKPOINT_DIR}/step_{step}"
            os.makedirs(ckpt_path, exist_ok=True)

            model.save_pretrained(ckpt_path)
            tokenizer.save_pretrained(ckpt_path)

            print(f"Checkpoint saved at step {step}")

    epoch_loss = total_loss / len(loader)

    print(f"Epoch {epoch+1} loss:", epoch_loss)

    ############################################
    # SAVE CHECKPOINT AFTER EACH EPOCH
    ############################################

    epoch_path = f"{CHECKPOINT_DIR}/epoch_{epoch+1}"
    os.makedirs(epoch_path, exist_ok=True)

    model.save_pretrained(epoch_path)
    tokenizer.save_pretrained(epoch_path)

    print(f"Epoch {epoch+1} checkpoint saved")

In [ ]:
FINAL_DIR = "/kaggle/working/final_model"
os.makedirs(FINAL_DIR, exist_ok=True)

model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Final model saved.")

In [ ]:
vowels = "aeiouy"

def fallback_syllables(word):

    word = word.lower()

    count = 0
    prev = False

    for c in word:

        if c in vowels:

            if not prev:
                count += 1

            prev = True

        else:
            prev = False

    if word.endswith("e"):
        count -= 1

    return max(1,count)


def syllables(word):

    word = word.lower()

    if word in cmu_dict:

        return max(
            [sum(1 for p in pron if str.isdigit(p[-1])) for pron in cmu_dict[word]]
        )

    return fallback_syllables(word)

In [ ]:
def generate_lyrics(emotion,meter=8,max_tokens=40):

    model.eval()

    prompt = f"<{emotion}>"

    input_ids = tokenizer.encode(prompt,return_tensors="pt").to(DEVICE)

    generated = input_ids

    words = []

    syllable_count = 0

    for _ in range(max_tokens):

        outputs = model(generated)

        logits = outputs.logits[:,-1,:]

        probs = torch.softmax(logits,dim=-1)

        topk = torch.topk(probs,50)

        candidates = topk.indices[0]

        valid = []

        for token in candidates:

            word = tokenizer.decode(token).strip()

            if len(word)==0:
                continue

            s = syllables(word)

            if syllable_count + s <= meter:

                valid.append(token)

        if len(valid)==0:
            break

        next_token = valid[torch.randint(len(valid),(1,))]

        generated = torch.cat(
            (generated,next_token.unsqueeze(0).unsqueeze(0)),
            dim=1
        )

        word = tokenizer.decode(next_token)

        words.append(word)

        syllable_count += syllables(word)

        if syllable_count >= meter:
            break

    return " ".join(words)